# 1 — VMD: the solver, and whether it converged

This notebook is about the decomposition itself. No classifier, no features: just the
question *does this thing work, and do I believe the modes it produces?*

---

## What VMD does, in plain words

An ECG trace is several things added together: a slow drifting baseline (the patient
breathing, the electrode settling), the heartbeat itself, and high-frequency muscle
noise and mains hum. You can see all of them at once in the raw trace, which is
exactly the problem — they are tangled.

**Variational Mode Decomposition splits one signal into `K` component signals that add
back up to the original**, with one extra demand: each component must be *narrow in
frequency*. Not "below 5 Hz" — you don't tell it where the bands are. It finds each
component's centre frequency itself, by asking: what set of `K` narrow-band signals,
added together, reconstructs this trace while being as narrow as possible?

Those components are the **IMFs** (intrinsic mode functions), and on ECG they come out
roughly as: mode 1 = baseline wander, middle modes = the QRS complex and T wave,
top modes = noise and high-frequency detail. That ordering is not imposed — it is
what the data does.

The knob that controls "as narrow as possible" is **alpha**. High alpha means very
narrow modes; low alpha lets each mode spread out and carry more of the waveform.

**Why not just use a band-pass filter?** A filter has fixed edges you chose in advance.
VMD moves its bands to wherever the energy actually is, per segment. A patient with a
60 bpm heart rate and one with 110 bpm get different band placements automatically.

**Why not EMD?** Empirical Mode Decomposition peels off components one at a time by
a recursive sifting procedure. It has no objective function, no convergence theory,
and suffers *mode mixing* — one component ends up containing two unrelated rhythms.
VMD solves all `K` modes simultaneously against a stated optimisation problem, which
is why it is stabler and why we can ask "did it converge?" at all.

In [ ]:
# --- Bootstrap: works unchanged in VS Code, plain Jupyter, and Google Colab ----------
import os, sys, subprocess

def _bootstrap():
    """Make `import ecgvmd` work, wherever this notebook is being run."""
    try:
        import ecgvmd  # noqa: F401
        return os.path.dirname(os.path.dirname(os.path.abspath(ecgvmd.__file__)))
    except ImportError:
        pass

    # walk up from the cwd looking for the package
    d = os.path.abspath(os.getcwd())
    for _ in range(4):
        if os.path.isdir(os.path.join(d, "ecgvmd")):
            sys.path.insert(0, d); return d
        d = os.path.dirname(d)

    # Colab: try Drive, then fall back to an upload
    for cand in ("/content/test-ecg-training", "/content/drive/MyDrive/test-ecg-training"):
        if os.path.isdir(os.path.join(cand, "ecgvmd")):
            sys.path.insert(0, cand); return cand
    try:
        from google.colab import files
        print("Upload ecgvmd_bundle.zip (make it with: python make_colab_bundle.py)")
        up = files.upload()
        import zipfile
        with zipfile.ZipFile(next(iter(up))) as z:
            z.extractall("/content")
        sys.path.insert(0, "/content"); return "/content"
    except ImportError:
        raise ImportError("Could not locate the ecgvmd package. Run this notebook from "
                          "the project root, or upload ecgvmd_bundle.zip in Colab.")

IN_COLAB = "google.colab" in sys.modules or os.path.isdir("/content")
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "-q", "install",
                    "numpy", "scipy", "scikit-learn", "pandas", "matplotlib"], check=False)

ROOT = _bootstrap()
os.chdir(ROOT)
print("project root:", ROOT)
print("python      :", sys.version.split()[0])

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import ecgvmd as E
from ecgvmd import CFG

%matplotlib inline
plt.rcParams.update({
    "figure.dpi": 110, "font.size": 9, "axes.grid": True, "grid.alpha": 0.25,
    "axes.spines.top": False, "axes.spines.right": False, "figure.facecolor": "white",
    "axes.titlesize": 10, "legend.frameon": False,
})

print("ecgvmd", E.__version__)
print(CFG.summary())

## 1.1 Does the solver actually solve?

The honest test is the synthetic signal from the original paper: three pure cosines at
3, 12 and 30 Hz with known amplitudes. If VMD is working, it recovers those three
frequencies without being told them, and the sum of the modes reproduces the input.

We run **both** implementations — the readable reference loop and the vectorised batch
solver — because the fast one is the one that will process 21,000 segments and it needs
to be provably the same algorithm.

In [ ]:
fs_syn, N_syn = 1000.0, 1000
t_syn = np.arange(N_syn) / fs_syn
sig_syn = (np.cos(2*np.pi*3*t_syn)
           + 0.25 * np.cos(2*np.pi*12*t_syn)
           + 0.0625 * np.cos(2*np.pi*30*t_syn))

ref = E.vmd(sig_syn, alpha=2000, K=3, dc=False, fs=fs_syn)
bat = E.vmd_batch(sig_syn[None], alpha=2000, K=3, dc=False, fs=fs_syn)

print("expected centre frequencies : [ 3. 12. 30.] Hz")
print("reference solver recovered  :", np.round(ref.omega_hz, 3))
print("batch solver recovered      :", np.round(bat.omega_hz[0], 3))
print()
print(f"max |reference - batch|      : {np.abs(ref.modes - bat.modes[0]).max():.3e}")
print(f"residual ENERGY fraction     : {float(ref.residual_energy_fraction(sig_syn)):.3e}")
print()
print(ref.report("reference"))

Both solvers land on the right frequencies and agree with each other. The residual is
at the level of floating-point noise, so the three modes really do add back up to the
input.

> **A note on residuals that trips people up.** `residual_norm_ratio` is
> $\lVert x - \sum_k u_k\rVert / \lVert x\rVert$. The fraction of *energy* left
> unexplained is its **square**. A norm ratio of 0.21 means **4.3%** of the energy is
> missing, not 21%. Both are exposed on `VMDResult` precisely because mixing them up
> overstates the error by about five-fold.

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(10, 6), sharex=True)
axes[0].plot(t_syn, sig_syn, lw=0.9, color="0.2")
axes[0].set_ylabel("input")
axes[0].set_title("VMD of the synthetic three-tone signal", loc="left")
for k in range(3):
    axes[k+1].plot(t_syn, ref.modes[k], lw=0.9, color=plt.cm.viridis(k / 3))
    axes[k+1].set_ylabel(f"$u_{k+1}$")
    axes[k+1].text(0.99, 0.85, f"{ref.omega_hz[k]:.2f} Hz", ha="right",
                   transform=axes[k+1].transAxes, fontsize=8, color="0.4")
axes[-1].set_xlabel("time (s)")
fig.tight_layout(); plt.show()

In [ ]:
# The centre frequencies are *found*, not given. This is the search doing its work.
fig, ax = plt.subplots(figsize=(7, 3.2))
for k in range(3):
    ax.plot(ref.omega_hist[:, k] * fs_syn, lw=1.2, color=plt.cm.viridis(k / 3),
            label=f"$\\omega_{k+1}$")
for f_true in (3, 12, 30):
    ax.axhline(f_true, ls=":", color="0.6", lw=0.8)
ax.set_xlabel("ADMM iteration"); ax.set_ylabel("centre frequency (Hz)")
ax.set_title("Centre frequencies converging from a uniform initialisation", loc="left")
ax.legend(); fig.tight_layout(); plt.show()

## 1.2 What `alpha` does to a real ECG segment

Now on real data. The same segment, decomposed at three very different bandwidth
penalties. This is the single most consequential parameter in the whole pipeline.

In [ ]:
ds = E.load_ecgdata()
print(ds)

demo_cfg = CFG.replace(n_per_record=4)
W, y, g = E.segment(ds, demo_cfg)
demo = W[np.where(y == "NSR")[0][3]]
t = np.arange(len(demo)) / CFG.fs
print("demo window:", demo.shape, "| class NSR")

In [ ]:
alphas_show = [5.0, 2000.0, 100000.0]
K_show = 5
fig, axes = plt.subplots(K_show + 1, len(alphas_show), figsize=(12, 7),
                         sharex=True, sharey="row")
for c, a in enumerate(alphas_show):
    r = E.vmd(demo, alpha=a, K=K_show, dc=True, fs=CFG.fs, max_iter=500)
    axes[0, c].plot(t, demo, lw=0.7, color="0.2")
    axes[0, c].set_title(f"alpha = {a:g}   ({r.iters[0]} iters, "
                         f"resid E = {float(r.residual_energy_fraction(demo)):.2%})",
                         loc="left", fontsize=9)
    for k in range(K_show):
        axes[k+1, c].plot(t, r.modes[k], lw=0.7, color=plt.cm.viridis(k / K_show))
        axes[k+1, c].text(0.99, 0.75, f"{r.omega_hz[k]:.1f} Hz", ha="right", fontsize=7,
                          color="0.45", transform=axes[k+1, c].transAxes)
axes[0, 0].set_ylabel("signal")
for k in range(K_show):
    axes[k+1, 0].set_ylabel(f"$u_{k+1}$")
for c in range(len(alphas_show)):
    axes[-1, c].set_xlabel("time (s)")
fig.suptitle("Same ECG window, three bandwidth penalties", y=1.0, fontsize=11)
fig.tight_layout(); plt.show()

Read the three columns as a trade-off:

* **alpha = 5** — modes are wide. Each one carries a recognisable chunk of ECG
  morphology. Reconstruction is near-perfect. But the modes overlap in frequency, so
  "mode 3" is not a clean band.
* **alpha = 2000** — the default. Modes are narrow and well separated, and the QRS
  complex is visibly split across several of them.
* **alpha = 100000** — modes are almost pure tones. The decomposition has become a
  Fourier analysis with extra steps, and a lot of the waveform is left in the residual.

## 1.3 The convergence question — and why it goes first

Here is the trap the earlier notebooks fell into.

ADMM is an iterative solver. It stops when the modes stop changing (`tol`) **or** when
it runs out of patience (`max_iter`). If it stops for the second reason, you are not
looking at the VMD solution — you are looking at wherever the iteration happened to be
when the clock ran out.

The earlier work found that **alpha = 5 classifies better than alpha = 2000**. But it
also turns out that at alpha = 5 essentially *every* segment exhausts `max_iter = 500`.
So the finding was ambiguous: is a wide-band decomposition genuinely better for ECG, or
is a *half-solved* decomposition better by accident?

Note the direction, because it is counter-intuitive: **lower alpha converges more
slowly, not faster.** With a weak bandwidth penalty the modes are poorly separated and
the centre frequencies keep sliding around each other for many more iterations.

In [ ]:
# The capped fraction is a first-class property of every result, so it cannot be missed.
probe = W[:96]
print(f"{'alpha':>8} {'max_iter':>9} {'capped':>8} {'median iters':>13} {'resid E':>9}")
print("-" * 52)
for a in [5.0, 50.0, 200.0, 2000.0, 8000.0]:
    for mi in [500, 2000]:
        r = E.vmd_batch(probe, alpha=a, K=CFG.K, dc=True, tol=CFG.tol,
                        max_iter=mi, fs=CFG.fs)
        print(f"{a:8g} {mi:9d} {100*r.capped_fraction:7.1f}% "
              f"{np.median(r.iters):13.0f} "
              f"{float(r.residual_energy_fraction(probe).mean()):8.2%}")

## 1.4 The settled answer

`scripts/alpha_sweep.py` runs the full experiment: both segmentation modes, six alphas,
at `max_iter = 500` and `max_iter = 2000`, scoring each with record-wise cross-validated
macro-F1. Run it once (it takes a while), then this cell reads the result.

```bash
python scripts/alpha_sweep.py            # full
python scripts/alpha_sweep.py --fast     # quick sanity version
```

In [ ]:
sweep_path = "results/alpha_sweep.csv"
if os.path.exists(sweep_path):
    sw = pd.read_csv(sweep_path)
    show = sw[["seg_mode", "alpha", "max_iter", "capped_frac", "median_iters",
               "resid_energy_frac", "segment_macroF1", "record_acc"]]
    print(show.to_string(index=False, float_format=lambda v: f"{v:.4f}"))
else:
    sw = None
    print(f"{sweep_path} not found - run  python scripts/alpha_sweep.py  first.")

In [ ]:
if sw is not None:
    fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.4))
    for mode, ls in [("fixed", "-"), ("beat", "--")]:
        for mi, col in [(500, "#c0392b"), (2000, "#2471a3")]:
            s = sw[(sw.seg_mode == mode) & (sw.max_iter == mi)].sort_values("alpha")
            if not len(s):
                continue
            lab = f"{mode}, max_iter={mi}"
            axes[0].semilogx(s.alpha, 100*s.capped_frac, ls, color=col, marker="o", ms=3, label=lab)
            axes[1].semilogx(s.alpha, 100*s.resid_energy_frac, ls, color=col, marker="o", ms=3, label=lab)
            axes[2].semilogx(s.alpha, s.segment_macroF1, ls, color=col, marker="o", ms=3, label=lab)
    axes[0].set_ylabel("% capped at max_iter"); axes[0].set_title("did it converge?", loc="left")
    axes[1].set_ylabel("residual energy (%)"); axes[1].set_title("what was left over?", loc="left")
    axes[2].set_ylabel("macro-F1 (record-wise CV)"); axes[2].set_title("does it classify?", loc="left")
    for ax in axes:
        ax.set_xlabel("alpha")
    axes[2].legend(fontsize=7)
    fig.tight_layout(); plt.show()

**How to read the three panels together.**

The left panel is the credibility check: any point high on that axis is a decomposition
that never finished, and its position in the right panel means little on its own.

**The measured answer** (fixed-grid windows, K=8, 1620 windows, all 162 records,
record-wise CV):

| alpha | % capped @500 | macro-F1 @500 | % capped @2000 | macro-F1 @2000 | residual energy |
|---:|---:|---:|---:|---:|---:|
| 5 | 100.0 | **0.8075** | 99.4 | **0.8010** | 0.00% |
| 50 | 99.5 | 0.7978 | 64.6 | 0.7843 | 0.01% |
| 200 | 93.7 | 0.7614 | 15.3 | 0.7739 | 0.08% |
| 500 | 75.6 | 0.7804 | 3.4 | 0.7573 | 0.32% |
| 2000 | 34.2 | 0.7861 | **0.6** | 0.7762 | 1.86% |
| 8000 | 21.2 | 0.7225 | **0.0** | 0.7233 | 11.11% |

Three things follow.

1. **alpha=5 never converges, and it does not matter.** Quadrupling the budget leaves it
   still 99.4% capped and moves macro-F1 only from 0.8075 to 0.8010. The score does not
   depend on where you truncate, so the earlier "alpha=5 is best" finding is *real*, not an
   artefact of stopping early. But the modes it produces are defined by `max_iter`, so that
   number is part of the method and has to be reported alongside it.

2. **Only the low end is stuck.** With `max_iter=2000`, alpha=200 falls to 15% capped,
   alpha=2000 to 0.6%, alpha=8000 to zero. A genuinely converged decomposition *is*
   available on this data — it just is not at alpha=5.

3. **Converging harder does not help classification.** The best fully converged setting is
   alpha=2000 at 0.7762, still 0.025 below unconverged alpha=5. And letting alpha=2000 run
   to convergence moves it 0.7861 → 0.7762 — slightly *worse*, though that gap is within
   noise on 162 records. Convergence buys interpretability, not accuracy.

**What to set.** alpha=5 if you want the best score and are willing to report `max_iter` as
part of the method; alpha=2000 with `max_iter=2000` if you want a decomposition you can
defend as solved. The shipped default is alpha=2000 / `max_iter=500`, a middle position —
change it in the config cell of notebook 2, since every downstream number inherits it.

## 1.5 Cost

The batched solver is what makes the full dataset affordable. This is the measurement,
not a claim.

In [ ]:
import time
bench = W[:64]
t0 = time.time()
for row in bench[:8]:
    E.vmd(row, alpha=CFG.alpha, K=CFG.K, dc=True, fs=CFG.fs)
per_ref = (time.time() - t0) / 8

t0 = time.time()
E.vmd_batch(bench, alpha=CFG.alpha, K=CFG.K, dc=True, fs=CFG.fs)
per_bat = (time.time() - t0) / len(bench)

n_full = 162 * (65536 // CFG.seg_len)
print(f"reference : {1000*per_ref:7.1f} ms/segment  ->  {n_full*per_ref/60:6.1f} min for all {n_full}")
print(f"batched   : {1000*per_bat:7.1f} ms/segment  ->  {n_full*per_bat/60:6.1f} min for all {n_full}")
print(f"speed-up  : {per_ref/per_bat:.1f}x")

---

**Next:** `02_imf_features.ipynb` turns the modes into the IMF feature matrix.